# Evolution of the LSTM temperature-forecasting models

This notebook reconstructs and explains the four supplied experiment summaries. Each section contains a complete Keras builder, and the final section trains any or all architectures directly from the chronological splits produced by `weather_main.py`.

The saved summaries reveal four distinct architectures. The **2026-07-01** model is a fixed-horizon stacked-LSTM regressor, while **2026-07-23** is a 72-to-24-hour residual forecaster built around the previous day's temperature curve.

| Experiment | Input per sample | Prediction | Main design | Parameters |
|---|---:|---:|---|---:|
| 2026-07-01 | 24 hours × 10 features | one temperature, 3 hours ahead | stacked LSTM | 33,238 |
| 2026-07-23 | 72 hours × 11 features | temperatures for hours 1–24 | LSTM correction to previous-day baseline | 20,287 |
| 2026-07-27 | 72 hours × 11 features | temperatures for hours 1–24 | CNN + LSTM + self-attention + previous-day residual | 212,232 |
| 2026-08-02 | 72 hours × 11 features | temperatures for hours 1–24 | compact encoder + self/cross-attention + learned baseline blend | 100,195 |

`None` in a model summary is the batch dimension. For example, `(None, 72, 11)` means any batch size, 72 hourly timesteps, and 11 features per hour.


## 1. Shared setup and feature contracts

The early model does **not** receive past temperature as an input feature; temperature is only the target. The later 24-hour models add past temperature to the feature tensor, which enables previous-day, persistence, and trend baselines without target leakage—only observations available before the forecast origin are used.


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model

tf.keras.utils.set_random_seed(21)

EARLY_FEATURE_COLUMNS = [
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation",
    "is_day",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

FEATURE_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation",
    "is_day",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

TARGET_COLUMN = "temperature_2m"
INPUT_HOURS = 72
OUTPUT_HOURS = 24

TEMPERATURE_FEATURE_INDEX = FEATURE_COLUMNS.index(TARGET_COLUMN)
HOUR_SIN_INDEX = FEATURE_COLUMNS.index("hour_sin")
HOUR_COS_INDEX = FEATURE_COLUMNS.index("hour_cos")
DAY_SIN_INDEX = FEATURE_COLUMNS.index("day_sin")
DAY_COS_INDEX = FEATURE_COLUMNS.index("day_cos")


def validate_training_tensor(X_train, hours, feature_count):
    if not isinstance(X_train, np.ndarray) or X_train.ndim != 3:
        raise ValueError("X_train must be a 3-D NumPy array: samples × hours × features.")
    if X_train.shape[1:] != (hours, feature_count):
        raise ValueError(
            f"Expected (*, {hours}, {feature_count}); received {X_train.shape}."
        )
    if not np.isfinite(X_train).all():
        raise ValueError("X_train contains NaN or infinite values.")


In [4]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name.lower() == "notebooks"
    else CURRENT_DIR
)

if not (PROJECT_ROOT / "weather_main.py").exists():
    raise FileNotFoundError(
        f"Could not find weather_main.py under {PROJECT_ROOT}"
    )

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\lj200\weatherAI\Temperature_Prediction_DeepLearning


## 2. Standard Stacked LSTM and Residual Baseline

### 2026-07-01: Standard Stacked LSTM regression

This model consumes 24 hours of 10 non-temperature weather and calendar features and predicts one temperature 3 hours after the end of the window. It uses 64- and 32-unit LSTMs followed by 32- and 16-unit dense layers. Ordinary dropout is applied between stages. The model has 33,238 parameters, including 21 non-trainable normalization statistics.

### 2026-07-23: Residual Baseline

This model changes both the input and the forecast task. It consumes 72 hours of 11 features—including observed temperature—and predicts all 24 temperatures for the next day. A 48/24-unit LSTM encoder predicts a 24-value adjustment vector. That adjustment is added to the final 24 observed temperatures, which represent the same hours on the previous day. The network therefore learns the departure from a strong daily-cycle baseline instead of rebuilding the entire temperature curve from zero.

The residual model has 20,287 parameters and uses layer normalization, L2 regularization, dropout, gradient clipping, and Huber loss.


In [5]:
def build_model_2026_07_01(X_train):
    '''Builder for experiment 2026-07-01_17-14-36.'''
    validate_training_tensor(X_train, hours=24, feature_count=10)

    normalizer = layers.Normalization(axis=-1)
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(24, 10),
        name="weather_sequence",
    )
    x = layers.TimeDistributed(
        normalizer,
        name="time_distributed",
    )(inputs)

    x = layers.LSTM(
        64,
        return_sequences=True,
        name="lstm",
    )(x)
    x = layers.Dropout(0.20, name="dropout")(x)
    x = layers.LSTM(32, name="lstm_1")(x)
    x = layers.Dropout(0.20, name="dropout_1")(x)

    x = layers.Dense(32, activation="relu", name="dense")(x)
    x = layers.Dropout(0.20, name="dropout_2")(x)
    x = layers.Dense(16, activation="relu", name="dense_1")(x)
    outputs = layers.Dense(1, name="temperature_prediction")(x)

    model = Model(inputs=inputs, outputs=outputs, name="functional")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanSquaredError(name="mse"),
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


def build_model_2026_07_23(X_train, learning_rate=3e-4):
    '''Builder for experiment 2026-07-23_19-38-36 (Residual Baseline).'''
    validate_training_tensor(X_train, hours=72, feature_count=11)

    normalizer = layers.Normalization(
        axis=-1,
        name="feature_normalization",
    )
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(INPUT_HOURS, len(FEATURE_COLUMNS)),
        name="weather_sequence",
    )
    x = normalizer(inputs)

    x = layers.LSTM(
        48,
        return_sequences=True,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(1e-4),
        name="lstm_1",
    )(x)
    x = layers.LayerNormalization(name="lstm_1_normalization")(x)

    x = layers.LSTM(
        24,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(1e-4),
        name="lstm_2",
    )(x)
    x = layers.LayerNormalization(name="lstm_2_normalization")(x)

    x = layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_features",
    )(x)
    x = layers.Dropout(0.20, name="dense_dropout")(x)

    temperature_adjustment = layers.Dense(
        OUTPUT_HOURS,
        name="temperature_adjustment",
    )(x)
    previous_day_temperature = inputs[
        :,
        -OUTPUT_HOURS:,
        TEMPERATURE_FEATURE_INDEX,
    ]
    outputs = layers.Add(name="temperature_prediction")(
        [previous_day_temperature, temperature_adjustment]
    )

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="weather_lstm_24h",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        ),
        loss=tf.keras.losses.Huber(delta=1.5),
        metrics=[
            tf.keras.metrics.MeanSquaredError(name="mse"),
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


### What changed after the Residual Baseline?

The July 27 Attention-Augmented model retains the July 23 residual idea—previous-day temperature plus a learned adjustment—but adds convolutional feature extraction, a larger LSTM encoder, self-attention, richer history pooling, and a horizon-aware decoder. The Cross-Attention model later expands the baseline idea again by blending previous-day, persistence, and trend forecasts.


## 3. Experiment 2026-07-27: convolutional LSTM with self-attention

This model has five conceptual stages:

1. A causal convolution and a separable convolution find short local weather patterns. A residual connection protects the original convolutional representation.
2. A 96-unit LSTM models the ordered 72-hour history.
3. Four-head self-attention lets every historical hour compare itself with every other hour. A Transformer-style feed-forward block follows it.
4. Average, maximum, and latest-state pooling summarize the encoded history from complementary viewpoints.
5. A horizon-aware decoder predicts 24 corrections, which are added to the temperatures observed at the same hours on the previous day.

The learned horizon embedding is important: forecast hour 1 and forecast hour 24 should not use exactly the same decoder representation.


In [6]:
@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class PreviousDayTemperature(layers.Layer):
    '''Return the latest 24 observed temperatures as a forecast baseline.'''

    def __init__(
        self,
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.feature_index = int(feature_index)

    def call(self, inputs):
        return inputs[:, -self.output_hours :, self.feature_index]

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "feature_index": self.feature_index,
            }
        )
        return config


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class HorizonEmbedding(layers.Layer):
    '''Learn one embedding vector for each future hour.'''

    def __init__(self, output_hours=OUTPUT_HOURS, embedding_dim=8, **kwargs):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.embedding_dim = int(embedding_dim)
        self.embedding = layers.Embedding(
            input_dim=self.output_hours,
            output_dim=self.embedding_dim,
        )

    def call(self, inputs):
        indices = tf.range(self.output_hours)
        embedded = self.embedding(indices)[tf.newaxis, :, :]
        return tf.tile(embedded, [tf.shape(inputs)[0], 1, 1])

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "embedding_dim": self.embedding_dim,
            }
        )
        return config


def build_model_2026_07_27(X_train, learning_rate=5e-4):
    validate_training_tensor(X_train, hours=72, feature_count=11)

    normalizer = layers.Normalization(axis=-1, name="feature_normalization")
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(INPUT_HOURS, len(FEATURE_COLUMNS)),
        name="weather_sequence",
    )
    x = normalizer(inputs)

    causal = layers.Conv1D(
        64,
        kernel_size=5,
        padding="causal",
        activation="swish",
        kernel_regularizer=regularizers.l2(5e-5),
        name="causal_conv",
    )(x)
    local = layers.SeparableConv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="swish",
        depthwise_regularizer=regularizers.l2(5e-5),
        pointwise_regularizer=regularizers.l2(5e-5),
        name="local_weather_block",
    )(causal)
    local = layers.Dropout(0.10, name="conv_dropout")(local)
    x = layers.Add(name="conv_residual_add")([causal, local])
    x = layers.LayerNormalization(name="conv_normalization")(x)

    x = layers.LSTM(
        96,
        return_sequences=True,
        dropout=0.10,
        kernel_regularizer=regularizers.l2(5e-5),
        name="lstm_encoder",
    )(x)

    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=24,
        dropout=0.10,
        name="history_attention",
    )(x, x)
    x = layers.Add(name="attention_residual_add")([x, attention])
    x = layers.LayerNormalization(name="attention_normalization")(x)

    feed_forward = layers.Dense(
        192,
        activation="swish",
        name="attention_ff_1",
    )(x)
    feed_forward = layers.Dropout(
        0.10,
        name="attention_ff_dropout",
    )(feed_forward)
    feed_forward = layers.Dense(
        96,
        name="attention_ff_2",
    )(feed_forward)
    x = layers.Add(name="feed_forward_residual_add")([x, feed_forward])
    encoded = layers.LayerNormalization(
        name="encoder_output_normalization",
    )(x)

    latest = layers.Cropping1D(
        cropping=(INPUT_HOURS - 1, 0),
        name="latest_timestep_crop",
    )(encoded)
    average = layers.GlobalAveragePooling1D(
        name="average_context",
    )(encoded)
    maximum = layers.GlobalMaxPooling1D(
        name="maximum_context",
    )(encoded)
    latest = layers.Reshape((96,), name="latest_context")(latest)

    context = layers.Concatenate(name="combined_context")(
        [average, maximum, latest]
    )
    context = layers.Dense(
        160,
        activation="swish",
        kernel_regularizer=regularizers.l2(5e-5),
        name="forecast_features_1",
    )(context)
    context = layers.Dropout(0.20, name="forecast_dropout_1")(context)
    context = layers.Dense(
        80,
        activation="swish",
        kernel_regularizer=regularizers.l2(5e-5),
        name="forecast_features_2",
    )(context)
    context = layers.Dropout(0.10, name="forecast_dropout_2")(context)

    previous_day = PreviousDayTemperature(
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        name="previous_day_temperature",
    )(inputs)
    repeated_context = layers.RepeatVector(
        OUTPUT_HOURS,
        name="repeat_context",
    )(context)
    horizon = HorizonEmbedding(
        output_hours=OUTPUT_HOURS,
        embedding_dim=12,
        name="horizon_embedding",
    )(context)
    previous_day_expanded = layers.Reshape(
        (OUTPUT_HOURS, 1),
        name="previous_day_expanded",
    )(previous_day)

    decoder = layers.Concatenate(name="horizon_decoder_input")(
        [repeated_context, horizon, previous_day_expanded]
    )
    decoder = layers.Dense(
        64,
        activation="swish",
        name="decoder_dense_1",
    )(decoder)
    decoder = layers.Dropout(0.10, name="decoder_dropout")(decoder)
    decoder = layers.Dense(
        32,
        activation="swish",
        name="decoder_dense_2",
    )(decoder)
    adjustment = layers.Dense(
        1,
        name="temperature_adjustment_per_hour",
    )(decoder)
    adjustment = layers.Reshape(
        (OUTPUT_HOURS,),
        name="temperature_adjustment",
    )(adjustment)
    outputs = layers.Add(name="temperature_prediction")(
        [previous_day, adjustment]
    )

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="weather_lstm_attention_24h",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        ),
        loss=tf.keras.losses.Huber(),
        metrics=[
            tf.keras.metrics.MeanSquaredError(name="mse"),
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


## 4. Experiment 2026-08-02: cross-attentive multi-baseline model

The final supplied architecture is smaller than the July 27 model—100,195 instead of 212,232 parameters—but introduces more forecasting structure.

- Two LSTMs and self-attention create a 40-dimensional representation for every historical hour.
- Each of the 24 future hours becomes a separate query containing its horizon embedding, future calendar phase, and three simple temperature baselines.
- Cross-attention allows each future-hour query to inspect all 72 encoded historical hours directly. The model no longer has to recover all temporal detail from one pooled vector.
- A GRU decodes the 24 queries jointly, allowing neighboring forecast hours to remain coherent.
- Softmax weights blend previous-day, persistence, and damped-trend baselines separately at every horizon. A learned correction is then added to the blend.
- `LevelChangeHuber` penalizes both temperature-level error and error in hour-to-hour changes, with modestly higher weight on later horizons.

The following helpers reproduce the custom layers used by the attached `model.py`.


In [7]:
@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class TemperatureBaselines(layers.Layer):
    '''Previous-day, persistence, and damped recent-trend forecasts.'''

    def __init__(
        self,
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        trend_lookback=6,
        trend_decay_hours=8.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.feature_index = int(feature_index)
        self.trend_lookback = int(trend_lookback)
        self.trend_decay_hours = float(trend_decay_hours)

    def call(self, inputs):
        temperature = inputs[:, :, self.feature_index]
        previous_day = temperature[:, -self.output_hours :]

        latest = temperature[:, -1:]
        persistence = tf.repeat(latest, repeats=self.output_hours, axis=1)

        earlier = temperature[
            :, -(self.trend_lookback + 1) : -self.trend_lookback
        ]
        slope_per_hour = (latest - earlier) / tf.cast(
            self.trend_lookback,
            inputs.dtype,
        )
        horizon = tf.cast(
            tf.range(1, self.output_hours + 1)[tf.newaxis, :],
            inputs.dtype,
        )
        damping = tf.exp(
            -horizon / tf.cast(self.trend_decay_hours, inputs.dtype)
        )
        trend = persistence + slope_per_hour * horizon * damping
        return tf.stack([previous_day, persistence, trend], axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "feature_index": self.feature_index,
                "trend_lookback": self.trend_lookback,
                "trend_decay_hours": self.trend_decay_hours,
            }
        )
        return config


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class FutureCalendarFeatures(layers.Layer):
    '''Rotate the last cyclical time features into the next 24 hours.'''

    def __init__(
        self,
        output_hours=OUTPUT_HOURS,
        hour_sin_index=HOUR_SIN_INDEX,
        hour_cos_index=HOUR_COS_INDEX,
        day_sin_index=DAY_SIN_INDEX,
        day_cos_index=DAY_COS_INDEX,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.hour_sin_index = int(hour_sin_index)
        self.hour_cos_index = int(hour_cos_index)
        self.day_sin_index = int(day_sin_index)
        self.day_cos_index = int(day_cos_index)

    def call(self, inputs):
        dtype = inputs.dtype
        horizon = tf.cast(
            tf.range(1, self.output_hours + 1)[tf.newaxis, :],
            dtype,
        )

        last_hour_sin = inputs[:, -1, self.hour_sin_index][:, tf.newaxis]
        last_hour_cos = inputs[:, -1, self.hour_cos_index][:, tf.newaxis]
        last_day_sin = inputs[:, -1, self.day_sin_index][:, tf.newaxis]
        last_day_cos = inputs[:, -1, self.day_cos_index][:, tf.newaxis]

        hour_angle = horizon * tf.cast(2.0 * np.pi / 24.0, dtype)
        hour_cos_rotation = tf.cos(hour_angle)
        hour_sin_rotation = tf.sin(hour_angle)
        future_hour_sin = (
            last_hour_sin * hour_cos_rotation
            + last_hour_cos * hour_sin_rotation
        )
        future_hour_cos = (
            last_hour_cos * hour_cos_rotation
            - last_hour_sin * hour_sin_rotation
        )

        day_angle = horizon * tf.cast(2.0 * np.pi / (24.0 * 365.25), dtype)
        day_cos_rotation = tf.cos(day_angle)
        day_sin_rotation = tf.sin(day_angle)
        future_day_sin = (
            last_day_sin * day_cos_rotation
            + last_day_cos * day_sin_rotation
        )
        future_day_cos = (
            last_day_cos * day_cos_rotation
            - last_day_sin * day_sin_rotation
        )

        return tf.stack(
            [
                future_hour_sin,
                future_hour_cos,
                future_day_sin,
                future_day_cos,
            ],
            axis=-1,
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "hour_sin_index": self.hour_sin_index,
                "hour_cos_index": self.hour_cos_index,
                "day_sin_index": self.day_sin_index,
                "day_cos_index": self.day_cos_index,
            }
        )
        return config


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class WeightedBaseline(layers.Layer):
    '''Apply learned per-hour softmax weights to baseline candidates.'''

    def call(self, inputs):
        baselines, weights = inputs
        return tf.reduce_sum(baselines * weights, axis=-1)


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class LevelChangeHuber(tf.keras.losses.Loss):
    '''Huber loss for forecast levels plus adjacent-hour changes.'''

    def __init__(
        self,
        delta=1.0,
        change_weight=0.25,
        late_horizon_weight=0.30,
        name="level_change_huber",
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.delta = float(delta)
        self.change_weight = float(change_weight)
        self.late_horizon_weight = float(late_horizon_weight)

    def _elementwise_huber(self, error):
        error = tf.convert_to_tensor(error)
        absolute_error = tf.abs(error)
        delta = tf.cast(self.delta, error.dtype)
        quadratic = tf.minimum(absolute_error, delta)
        linear = absolute_error - quadratic
        return 0.5 * tf.square(quadratic) + delta * linear

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        level_loss = self._elementwise_huber(y_pred - y_true)

        horizon = tf.linspace(
            tf.cast(0.0, y_pred.dtype),
            tf.cast(1.0, y_pred.dtype),
            OUTPUT_HOURS,
        )
        horizon_weights = 1.0 + tf.cast(
            self.late_horizon_weight,
            y_pred.dtype,
        ) * horizon
        weighted_level_loss = tf.reduce_sum(
            level_loss * horizon_weights,
            axis=-1,
        ) / tf.reduce_sum(horizon_weights)

        true_change = y_true[:, 1:] - y_true[:, :-1]
        predicted_change = y_pred[:, 1:] - y_pred[:, :-1]
        change_loss = tf.reduce_mean(
            self._elementwise_huber(predicted_change - true_change),
            axis=-1,
        )
        return weighted_level_loss + tf.cast(
            self.change_weight,
            y_pred.dtype,
        ) * change_loss

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "delta": self.delta,
                "change_weight": self.change_weight,
                "late_horizon_weight": self.late_horizon_weight,
            }
        )
        return config


def make_optimizer(learning_rate=2e-4, weight_decay=1e-4):
    '''Use AdamW+EMA when supported; retain an Adam fallback.'''
    try:
        return tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            clipnorm=1.0,
            use_ema=True,
            ema_momentum=0.99,
        )
    except (AttributeError, TypeError):
        return tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        )


In [8]:
def build_model_2026_08_02(
    X_train,
    learning_rate=2e-4,
    weight_decay=1e-4,
):
    validate_training_tensor(X_train, hours=72, feature_count=11)

    normalizer = layers.Normalization(axis=-1, name="feature_normalization")
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(INPUT_HOURS, len(FEATURE_COLUMNS)),
        name="weather_sequence",
    )
    x = normalizer(inputs)

    x = layers.Conv1D(
        48,
        kernel_size=5,
        padding="same",
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="local_pattern_conv",
    )(x)
    x = layers.LayerNormalization(name="conv_normalization")(x)
    x = layers.SpatialDropout1D(
        0.12,
        name="conv_spatial_dropout",
    )(x)

    x = layers.LSTM(
        64,
        return_sequences=True,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(8e-5),
        recurrent_regularizer=regularizers.l2(4e-5),
        name="history_lstm_1",
    )(x)
    x = layers.LSTM(
        40,
        return_sequences=True,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(8e-5),
        recurrent_regularizer=regularizers.l2(4e-5),
        name="history_lstm_2",
    )(x)

    self_attention = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=20,
        dropout=0.15,
        name="history_self_attention",
    )(x, x)
    x = layers.Add(name="history_attention_residual")([x, self_attention])
    encoded_history = layers.LayerNormalization(
        name="encoded_history",
    )(x)

    average = layers.GlobalAveragePooling1D(
        name="average_context",
    )(encoded_history)
    maximum = layers.GlobalMaxPooling1D(
        name="maximum_context",
    )(encoded_history)
    latest = layers.Cropping1D(
        cropping=(INPUT_HOURS - 1, 0),
        name="latest_context_crop",
    )(encoded_history)
    latest = layers.Reshape((40,), name="latest_context")(latest)

    context = layers.Concatenate(name="combined_context")(
        [average, maximum, latest]
    )
    context = layers.Dense(
        80,
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="context_dense",
    )(context)
    context = layers.Dropout(0.25, name="context_dropout")(context)

    horizon_embedding = HorizonEmbedding(
        output_hours=OUTPUT_HOURS,
        embedding_dim=12,
        name="horizon_embedding",
    )(context)
    future_calendar = FutureCalendarFeatures(
        output_hours=OUTPUT_HOURS,
        name="future_calendar",
    )(inputs)
    baselines = TemperatureBaselines(
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        name="temperature_baselines",
    )(inputs)

    forecast_queries = layers.Concatenate(
        name="forecast_query_features",
    )([horizon_embedding, future_calendar, baselines])
    forecast_queries = layers.Dense(
        40,
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="forecast_query_projection",
    )(forecast_queries)

    cross_attention = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=20,
        dropout=0.15,
        name="forecast_to_history_attention",
    )(
        query=forecast_queries,
        value=encoded_history,
        key=encoded_history,
    )
    forecast_queries = layers.Add(name="cross_attention_residual")(
        [forecast_queries, cross_attention]
    )
    forecast_queries = layers.LayerNormalization(
        name="cross_attention_normalization",
    )(forecast_queries)

    repeated_context = layers.RepeatVector(
        OUTPUT_HOURS,
        name="repeat_global_context",
    )(context)
    decoder_input = layers.Concatenate(name="decoder_input")(
        [
            forecast_queries,
            repeated_context,
            future_calendar,
            baselines,
        ]
    )
    decoder = layers.GRU(
        48,
        return_sequences=True,
        dropout=0.12,
        kernel_regularizer=regularizers.l2(8e-5),
        recurrent_regularizer=regularizers.l2(4e-5),
        name="forecast_decoder_gru",
    )(decoder_input)
    decoder = layers.Dense(
        40,
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="decoder_dense",
    )(decoder)
    decoder = layers.Dropout(0.12, name="decoder_dropout")(decoder)

    baseline_weights = layers.Dense(
        3,
        activation="softmax",
        name="baseline_weights",
    )(decoder)
    blended_baseline = WeightedBaseline(
        name="blended_baseline",
    )([baselines, baseline_weights])

    correction = layers.Dense(
        1,
        kernel_regularizer=regularizers.l2(5e-5),
        name="temperature_correction",
    )(decoder)
    correction = layers.Reshape(
        (OUTPUT_HOURS,),
        name="temperature_correction_vector",
    )(correction)
    outputs = layers.Add(name="temperature_prediction")(
        [blended_baseline, correction]
    )

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="weather_lstm_cross_attention_24h",
    )
    model.compile(
        optimizer=make_optimizer(learning_rate, weight_decay),
        loss=LevelChangeHuber(
            delta=1.0,
            change_weight=0.25,
            late_horizon_weight=0.30,
        ),
        metrics=[
            tf.keras.metrics.MeanSquaredError(name="mse"),
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


## 5. Rebuild and verify every supplied summary

This smoke test uses small synthetic arrays only to adapt each normalization layer and build the graphs. It does not train the models. The assertions compare each graph with the parameter count and output shape in its supplied `model_summary.txt`.

When using real data, pass only the training split to a builder so the normalization statistics do not absorb validation or test information.


In [9]:
rng = np.random.default_rng(21)
demo_24x10 = rng.normal(size=(32, 24, 10)).astype(np.float32)
demo_72x11 = rng.normal(size=(32, 72, 11)).astype(np.float32)

models = {
    "2026-07-01": build_model_2026_07_01(demo_24x10),
    "2026-07-23": build_model_2026_07_23(demo_72x11),
    "2026-07-27": build_model_2026_07_27(demo_72x11),
    "2026-08-02": build_model_2026_08_02(demo_72x11),
}

expected = {
    "2026-07-01": {"output_shape": (None, 1), "params": 33_238},
    "2026-07-23": {"output_shape": (None, 24), "params": 20_287},
    "2026-07-27": {"output_shape": (None, 24), "params": 212_232},
    "2026-08-02": {"output_shape": (None, 24), "params": 100_195},
}

print(f"{'Experiment':<12} {'Output shape':<18} {'Parameters':>12}")
print("-" * 46)
for experiment, model in models.items():
    actual_shape = tuple(model.output_shape)
    actual_params = model.count_params()
    print(f"{experiment:<12} {str(actual_shape):<18} {actual_params:>12,}")
    assert actual_shape == expected[experiment]["output_shape"]
    assert actual_params == expected[experiment]["params"]

print("\nAll four builders match their supplied summaries.")

# Release the four smoke-test graphs before real training.
del models, demo_24x10, demo_72x11
tf.keras.backend.clear_session()



Experiment   Output shape         Parameters
----------------------------------------------
2026-07-01   (None, 1)                33,238
2026-07-23   (None, 24)               20,287
2026-07-27   (None, 24)              212,232
2026-08-02   (None, 24)              100,195

All four builders match their supplied summaries.


## 6. Train every architecture from `weather_main.py` splits

Run `weather_main.py` from the project root first. It creates `data/splits/train.csv`, `dev.csv`, and `test.csv`; this section reads those files directly and keeps every window inside one city and one uninterrupted hourly segment.

The final cell runs all four registered architectures sequentially through the notebook's internal trainer. Each model receives its own input/output contract, and TensorFlow memory is cleared between runs.

Each completed run is written to:

`models/LSTM/experiments/<timestamp> (<architecture name>)/`

Every experiment contains:

- `metrics.json`
- `model_config.json`
- `model_summary.txt`
- `predictions.csv`
- `training_history.csv`
- `weather_lstm.keras`
- `figures/loss_curve.png`
- `figures/mae_curve.png`
- `figures/predicted_vs_actual_temperature.png`

The early Standard Stacked LSTM retains its original 24-hour input, 10-feature contract and predicts a single temperature three hours ahead. The other three models use the shared 72-hour-to-24-hour task. The saved predictions are long-form and include city, timestamp, forecast hour, actual temperature, predicted temperature, error, persistence, and previous-day baselines, so they can be reused for later error analysis.


In [10]:
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import gc
import json
import random
import shutil

import matplotlib.pyplot as plt
import pandas as pd


RANDOM_SEED = 21
DEFAULT_MAX_EPOCHS = 100
DEFAULT_BATCH_SIZE = 64
DEFAULT_STRIDE = 3


@dataclass(frozen=True)
class SequenceMetadata:
    city: str
    forecast_start: str


def set_reproducible_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def discover_project_root():
    '''Find the repository containing weather_main.py and data/splits.'''
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (
            (candidate / "weather_main.py").exists()
            and (candidate / "data" / "splits").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run weather_main.py first, then "
        "start this notebook from inside Temperature_Prediction_DeepLearning."
    )


def load_and_validate_split(path, required_features, split_name):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run weather_main.py to create the data splits."
        )

    dataframe = pd.read_csv(path)
    required = ["time", "city", TARGET_COLUMN, *required_features]
    missing = sorted(set(required) - set(dataframe.columns))
    if missing:
        raise ValueError(f"{split_name} is missing columns: {missing}")

    dataframe = dataframe.copy()
    dataframe["time"] = pd.to_datetime(dataframe["time"], errors="coerce")
    if dataframe["time"].isna().any():
        raise ValueError(f"{split_name} contains invalid timestamps.")

    numeric_columns = sorted(set([TARGET_COLUMN, *required_features]))
    for column in numeric_columns:
        dataframe[column] = pd.to_numeric(dataframe[column], errors="coerce")
    invalid = dataframe[numeric_columns].isna().sum()
    invalid = invalid[invalid > 0]
    if not invalid.empty:
        raise ValueError(
            f"{split_name} contains missing or non-numeric values:\n"
            f"{invalid.to_string()}"
        )

    dataframe[numeric_columns] = dataframe[numeric_columns].astype(np.float32)
    return dataframe


def contiguous_city_segments(dataframe):
    '''Yield city-safe segments and break at every gap larger than one hour.'''
    for city, city_df in dataframe.groupby("city", sort=True):
        city_df = city_df.sort_values("time").reset_index(drop=True)
        segment_ids = (
            city_df["time"].diff().ne(pd.Timedelta(hours=1)).cumsum()
        )
        for _, segment_df in city_df.groupby(segment_ids, sort=False):
            yield str(city), segment_df.reset_index(drop=True)


def create_model_sequences(dataframe, spec, stride=DEFAULT_STRIDE):
    '''Create the exact input/target contract for one registered model.'''
    if stride < 1:
        raise ValueError("stride must be at least 1.")

    input_hours = spec["input_hours"]
    output_hours = spec["output_hours"]
    forecast_horizon = spec["forecast_horizon"]
    feature_columns = spec["feature_columns"]

    X_parts = []
    y_parts = []
    persistence_parts = []
    previous_day_parts = []
    metadata = []

    for city, segment_df in contiguous_city_segments(dataframe):
        features = segment_df[feature_columns].to_numpy(dtype=np.float32)
        target = segment_df[TARGET_COLUMN].to_numpy(dtype=np.float32)

        if output_hours == 1:
            minimum_rows = input_hours + forecast_horizon
        else:
            minimum_rows = input_hours + output_hours
        if len(segment_df) < minimum_rows:
            continue

        number_of_starts = len(segment_df) - minimum_rows + 1
        for start in range(0, number_of_starts, stride):
            input_end = start + input_hours
            X_parts.append(features[start:input_end])

            if output_hours == 1:
                target_index = input_end + forecast_horizon - 1
                target_values = target[target_index : target_index + 1]
                forecast_start = segment_df.loc[target_index, "time"]
                previous_day_index = target_index - 24
                previous_day = target[
                    previous_day_index : previous_day_index + 1
                ]
            else:
                target_end = input_end + output_hours
                target_values = target[input_end:target_end]
                forecast_start = segment_df.loc[input_end, "time"]
                previous_day = target[input_end - output_hours : input_end]

            y_parts.append(target_values)
            persistence_parts.append(
                np.full(output_hours, target[input_end - 1], dtype=np.float32)
            )
            previous_day_parts.append(previous_day.astype(np.float32, copy=False))
            metadata.append(
                SequenceMetadata(
                    city=city,
                    forecast_start=pd.Timestamp(forecast_start).isoformat(),
                )
            )

    if not X_parts:
        raise ValueError(
            f"No sequences were created for {spec['display_name']}. "
            "Check that each city has a sufficiently long uninterrupted span."
        )

    return (
        np.asarray(X_parts, dtype=np.float32),
        np.asarray(y_parts, dtype=np.float32),
        metadata,
        {
            "persistence": np.asarray(persistence_parts, dtype=np.float32),
            "previous_day": np.asarray(previous_day_parts, dtype=np.float32),
        },
    )


In [11]:
MODEL_SPECS = {
    "standard_stacked": {
        "display_name": "Standard Stacked LSTM regression model",
        "builder": build_model_2026_07_01,
        "input_hours": 24,
        "output_hours": 1,
        "forecast_horizon": 3,
        "feature_columns": EARLY_FEATURE_COLUMNS,
        "expected_parameters": 33_238,
    },
    "residual_baseline": {
        "display_name": "Residual Baseline",
        "builder": build_model_2026_07_23,
        "input_hours": 72,
        "output_hours": 24,
        "forecast_horizon": 1,
        "feature_columns": FEATURE_COLUMNS,
        "expected_parameters": 20_287,
    },
    "attention_augmented_residual": {
        "display_name": "Attention-Augmented Residual LSTM",
        "builder": build_model_2026_07_27,
        "input_hours": 72,
        "output_hours": 24,
        "forecast_horizon": 1,
        "feature_columns": FEATURE_COLUMNS,
        "expected_parameters": 212_232,
    },
    "cross_attention": {
        "display_name": "Cross-Attention LSTM",
        "builder": build_model_2026_08_02,
        "input_hours": 72,
        "output_hours": 24,
        "forecast_horizon": 1,
        "feature_columns": FEATURE_COLUMNS,
        "expected_parameters": 100_195,
    },
}


def calculate_regression_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    error = predicted - actual
    absolute_error = np.abs(error)
    squared_error = np.square(error)

    return {
        "overall_mse": float(np.mean(squared_error)),
        "overall_mae": float(np.mean(absolute_error)),
        "overall_rmse": float(np.sqrt(np.mean(squared_error))),
        "overall_bias": float(np.mean(error)),
        "max_absolute_error": float(np.max(absolute_error)),
        "mae_by_forecast_hour": np.mean(absolute_error, axis=0).tolist(),
        "rmse_by_forecast_hour": np.sqrt(
            np.mean(squared_error, axis=0)
        ).tolist(),
        "bias_by_forecast_hour": np.mean(error, axis=0).tolist(),
    }


def calculate_city_metrics(actual, predicted, metadata):
    cities = np.asarray([item.city for item in metadata], dtype=object)
    results = {}
    for city in sorted(set(cities.tolist())):
        mask = cities == city
        results[city] = calculate_regression_metrics(
            actual[mask], predicted[mask]
        )
        results[city]["samples"] = int(mask.sum())
    return results


def save_prediction_table(
    path,
    actual,
    predicted,
    metadata,
    baselines,
    spec,
    chunk_samples=5_000,
):
    '''Write full long-form predictions without constructing one giant frame.'''
    output_hours = spec["output_hours"]
    single_hour_label = spec["forecast_horizon"]
    first_chunk = True

    for start in range(0, len(actual), chunk_samples):
        stop = min(start + chunk_samples, len(actual))
        number_of_samples = stop - start
        sample_ids = np.arange(start, stop)
        cities = np.asarray(
            [item.city for item in metadata[start:stop]], dtype=object
        )
        forecast_starts = pd.to_datetime(
            [item.forecast_start for item in metadata[start:stop]]
        )

        if output_hours == 1:
            forecast_hours = np.full(number_of_samples, single_hour_label)
            forecast_times = forecast_starts
        else:
            hour_offsets = np.tile(np.arange(output_hours), number_of_samples)
            forecast_hours = hour_offsets + 1
            forecast_times = (
                np.repeat(forecast_starts.to_numpy(), output_hours)
                + pd.to_timedelta(hour_offsets, unit="h")
            )

        actual_flat = actual[start:stop].reshape(-1)
        predicted_flat = predicted[start:stop].reshape(-1)
        error = predicted_flat - actual_flat

        frame = pd.DataFrame(
            {
                "sample": np.repeat(sample_ids, output_hours),
                "city": np.repeat(cities, output_hours),
                "forecast_time": forecast_times,
                "forecast_hour": forecast_hours,
                "actual_temperature_c": actual_flat,
                "predicted_temperature_c": predicted_flat,
                "error_c": error,
                "absolute_error_c": np.abs(error),
                "persistence_temperature_c": baselines["persistence"][
                    start:stop
                ].reshape(-1),
                "previous_day_temperature_c": baselines["previous_day"][
                    start:stop
                ].reshape(-1),
            }
        )
        frame.to_csv(
            path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
        )
        first_chunk = False


def save_training_figures(figure_dir, history, actual, predicted, best_epoch):
    figure_dir.mkdir(parents=True, exist_ok=True)
    history_dict = history.history
    epochs = np.arange(1, len(history_dict["mae"]) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history_dict["mse"], label="Training MSE")
    plt.plot(epochs, history_dict["val_mse"], label="Validation MSE")
    plt.axvline(
        best_epoch,
        color="gray",
        linestyle="--",
        linewidth=1,
        label=f"Selected epoch: {best_epoch}",
    )
    plt.title("Training vs Validation MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(figure_dir / "loss_curve.png", dpi=300)
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, history_dict["mae"], label="Training MAE")
    plt.plot(epochs, history_dict["val_mae"], label="Validation MAE")
    plt.axvline(
        best_epoch,
        color="gray",
        linestyle="--",
        linewidth=1,
        label=f"Selected epoch: {best_epoch}",
    )
    plt.title("Training vs Validation MAE")
    plt.xlabel("Epoch")
    plt.ylabel("Mean Absolute Error (°C)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(figure_dir / "mae_curve.png", dpi=300)
    plt.close()

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)
    if len(actual_flat) > 250_000:
        rng = np.random.default_rng(RANDOM_SEED)
        plot_indices = rng.choice(
            len(actual_flat), size=250_000, replace=False
        )
        actual_plot = actual_flat[plot_indices]
        predicted_plot = predicted_flat[plot_indices]
    else:
        actual_plot = actual_flat
        predicted_plot = predicted_flat

    lower = float(min(actual_plot.min(), predicted_plot.min()))
    upper = float(max(actual_plot.max(), predicted_plot.max()))
    slope, intercept = np.polyfit(actual_plot, predicted_plot, 1)

    plt.figure(figsize=(8, 7))
    density = plt.hexbin(
        actual_plot,
        predicted_plot,
        gridsize=65,
        bins="log",
        mincnt=1,
        cmap="Blues",
    )
    plt.plot(
        [lower, upper],
        [lower, upper],
        color="gray",
        linestyle="--",
        label="Identity",
    )
    plt.plot(
        [lower, upper],
        [intercept + slope * lower, intercept + slope * upper],
        color="red",
        linewidth=1.5,
        label=f"Fit (slope={slope:.3f})",
    )
    plt.colorbar(density, label="log10 count")
    plt.title("Predicted vs Actual Temperature")
    plt.xlabel("Actual Temperature (°C)")
    plt.ylabel("Predicted Temperature (°C)")
    plt.xlim(lower, upper)
    plt.ylim(lower, upper)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(
        figure_dir / "predicted_vs_actual_temperature.png",
        dpi=300,
    )
    plt.close()


In [12]:
def create_experiment_directory(project_root, display_name):
    model_root = project_root / "models" / "LSTM" / "experiments"
    model_root.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    candidate = model_root / f"{timestamp} ({display_name})"
    suffix = 2
    while candidate.exists():
        candidate = model_root / f"{timestamp} ({display_name}) {suffix}"
        suffix += 1
    candidate.mkdir(parents=True)
    return candidate


def train_one_model(
    model_key,
    *,
    max_epochs=DEFAULT_MAX_EPOCHS,
    batch_size=DEFAULT_BATCH_SIZE,
    stride=DEFAULT_STRIDE,
    early_stopping_patience=10,
):
    if model_key not in MODEL_SPECS:
        raise KeyError(
            f"Unknown model '{model_key}'. Choose from {list(MODEL_SPECS)}."
        )

    tf.keras.backend.clear_session()
    set_reproducible_seed()
    spec = MODEL_SPECS[model_key]
    project_root = discover_project_root()
    split_dir = project_root / "data" / "splits"

    print("=" * 80)
    print(f"Preparing {spec['display_name']}")
    print("=" * 80)

    train_df = load_and_validate_split(
        split_dir / "train.csv", spec["feature_columns"], "training split"
    )
    dev_df = load_and_validate_split(
        split_dir / "dev.csv", spec["feature_columns"], "development split"
    )
    test_df = load_and_validate_split(
        split_dir / "test.csv", spec["feature_columns"], "test split"
    )

    X_train, y_train, _, _ = create_model_sequences(train_df, spec, stride)
    X_dev, y_dev, _, _ = create_model_sequences(dev_df, spec, stride)
    X_test, y_test, test_metadata, test_baselines = create_model_sequences(
        test_df, spec, stride
    )

    print(f"Training samples    : {len(X_train):,}")
    print(f"Development samples : {len(X_dev):,}")
    print(f"Test samples        : {len(X_test):,}")
    print(f"Input shape         : {X_train.shape}")
    print(f"Target shape        : {y_train.shape}")

    experiment_dir = create_experiment_directory(
        project_root, spec["display_name"]
    )
    figure_dir = experiment_dir / "figures"
    model_path = experiment_dir / "weather_lstm.keras"

    model = spec["builder"](X_train)
    if model.count_params() != spec["expected_parameters"]:
        raise AssertionError(
            f"{spec['display_name']} has {model.count_params():,} parameters; "
            f"expected {spec['expected_parameters']:,}."
        )

    with (experiment_dir / "model_summary.txt").open(
        "w", encoding="utf-8"
    ) as summary_file:
        model.summary(print_fn=lambda line: summary_file.write(line + "\n"))

    callbacks = [
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path),
            monitor="val_mae",
            mode="min",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae",
            mode="min",
            factor=0.5,
            patience=max(2, early_stopping_patience // 3),
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae",
            mode="min",
            patience=early_stopping_patience,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_dev, y_dev),
        epochs=max_epochs,
        batch_size=batch_size,
        shuffle=False,
        callbacks=callbacks,
        verbose=1,
    )

    best_epoch = int(np.argmin(history.history["val_mae"])) + 1
    # Reload the checkpoint selected by validation MAE so the model artifact
    # and every test metric below refer to exactly the same weights.
    model = tf.keras.models.load_model(model_path)

    history_frame = pd.DataFrame(history.history)
    history_frame.insert(0, "epoch", np.arange(1, len(history_frame) + 1))
    history_frame.to_csv(
        experiment_dir / "training_history.csv", index=False
    )

    evaluation = {
        key: float(value)
        for key, value in model.evaluate(
            X_test, y_test, verbose=0, return_dict=True
        ).items()
    }
    predictions = np.asarray(model.predict(X_test, verbose=0), dtype=np.float32)
    predictions = predictions.reshape(-1, spec["output_hours"])

    test_metrics = calculate_regression_metrics(y_test, predictions)
    persistence_metrics = calculate_regression_metrics(
        y_test, test_baselines["persistence"]
    )
    previous_day_metrics = calculate_regression_metrics(
        y_test, test_baselines["previous_day"]
    )
    city_metrics = calculate_city_metrics(
        y_test, predictions, test_metadata
    )

    save_prediction_table(
        experiment_dir / "predictions.csv",
        y_test,
        predictions,
        test_metadata,
        test_baselines,
        spec,
    )
    save_training_figures(
        figure_dir,
        history,
        y_test,
        predictions,
        best_epoch,
    )

    metrics = {
        "model_key": model_key,
        "model_name": model.name,
        "architecture": spec["display_name"],
        "created_at": experiment_dir.name.split(" (")[0],
        "parameter_count": int(model.count_params()),
        "best_epoch": best_epoch,
        "test_mse": test_metrics["overall_mse"],
        "test_mae_c": test_metrics["overall_mae"],
        "test_rmse_c": test_metrics["overall_rmse"],
        "test_bias_c": test_metrics["overall_bias"],
        "keras_test_evaluation": evaluation,
        "improved_model": test_metrics,
        "test_metrics_by_city": city_metrics,
        "persistence_baseline": persistence_metrics,
        "previous_day_baseline": previous_day_metrics,
        "training": {
            "epochs_requested": int(max_epochs),
            "epochs_completed": int(len(history_frame)),
            "batch_size": int(batch_size),
            "stride": int(stride),
            "random_seed": RANDOM_SEED,
            "selection_metric": "val_mae",
        },
        "data_contract": {
            "input_hours": spec["input_hours"],
            "output_hours": spec["output_hours"],
            "forecast_horizon": spec["forecast_horizon"],
            "feature_columns": spec["feature_columns"],
            "target_column": TARGET_COLUMN,
            "training_samples": int(len(X_train)),
            "development_samples": int(len(X_dev)),
            "test_samples": int(len(X_test)),
        },
    }
    with (experiment_dir / "metrics.json").open(
        "w", encoding="utf-8"
    ) as metrics_file:
        json.dump(metrics, metrics_file, indent=2)

    # predict.py reads this configuration and, by default, the matching model
    # under models/LSTM/latest. Publishing both files together prevents the
    # predictor from pairing a new model with an older feature contract.
    config = {
        "model_name": model.name,
        "architecture": model_key,
        "input_hours": spec["input_hours"],
        "output_hours": spec["output_hours"],
        "forecast_horizon": spec["forecast_horizon"],
        "feature_columns": spec["feature_columns"],
        "target_column": TARGET_COLUMN,
        "created_at": metrics["created_at"],
        "best_epoch": best_epoch,
        "training_stride": int(stride),
        "batch_size": int(batch_size),
        "selection_metric": "val_mae",
        "random_seed": RANDOM_SEED,
    }
    config_path = experiment_dir / "model_config.json"
    with config_path.open("w", encoding="utf-8") as config_file:
        json.dump(config, config_file, indent=2)

    latest_dir = project_root / "models" / "LSTM" / "latest"
    latest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(model_path, latest_dir / "weather_lstm.keras")
    shutil.copy2(config_path, latest_dir / "model_config.json")

    print("\nTest results")
    print(f"MSE  : {test_metrics['overall_mse']:.4f}")
    print(f"MAE  : {test_metrics['overall_mae']:.4f} °C")
    print(f"RMSE : {test_metrics['overall_rmse']:.4f} °C")
    print(f"Saved to: {experiment_dir}")

    result = {
        "model_key": model_key,
        "experiment_dir": str(experiment_dir),
        "metrics": metrics,
    }

    del model, history, X_train, y_train, X_dev, y_dev, X_test, y_test
    tf.keras.backend.clear_session()
    gc.collect()
    return result


def train_models(model_keys, **training_options):
    results = []
    for model_key in model_keys:
        results.append(train_one_model(model_key, **training_options))
    return results


### Run training

The cell below trains all four models through the notebook's internal architecture registry. Do not pass `standard_stacked` or `residual_baseline` to the separate `train.py` command-line interface; that script currently serves only the two attention models. Restart the kernel and run every notebook cell in order so the TensorFlow settings, builders, and training helpers are defined.


In [13]:
# Use the notebook trainer so each architecture receives its correct data contract.
MODELS_TO_TRAIN = [
    "standard_stacked",
    "residual_baseline",
    #"attention_augmented_residual",
    #"cross_attention",
]

MAX_EPOCHS = 100
BATCH_SIZE = 64
WINDOW_STRIDE = 3
EARLY_STOPPING_PATIENCE = 10

if "train_models" not in globals() or "MODEL_SPECS" not in globals():
    raise RuntimeError(
        "Restart the kernel, then run all notebook cells above this one."
    )

unknown_models = sorted(set(MODELS_TO_TRAIN) - set(MODEL_SPECS))
if unknown_models:
    raise ValueError(
        f"Unknown notebook model key(s): {unknown_models}. "
        f"Choose from {list(MODEL_SPECS)}."
    )

training_results = train_models(
    MODELS_TO_TRAIN,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    stride=WINDOW_STRIDE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)

training_summary = pd.DataFrame(
    [
        {
            "model": item["metrics"]["architecture"],
            "test_mse": item["metrics"]["test_mse"],
            "test_mae_c": item["metrics"]["test_mae_c"],
            "test_rmse_c": item["metrics"]["test_rmse_c"],
            "experiment_dir": item["experiment_dir"],
        }
        for item in training_results
    ]
)

display(training_summary)


Preparing Standard Stacked LSTM regression model
Training samples    : 376,192
Development samples : 80,512
Test samples        : 80,512
Input shape         : (376192, 24, 10)
Target shape        : (376192, 1)


Epoch 1/100
5873/5878 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 49.7021 - mae: 5.2670 - mse: 49.7021 - rmse: 6.9038
Epoch 1: val_mae improved from None to 7.46868, saving model to c:\Users\lj200\weatherAI\Temperature_Prediction_DeepLearning\models\LSTM\experiments\2026-08-16_08-14-02 (Standard Stacked LSTM regression model)\weather_lstm.keras

Epoch 1: finished saving model to c:\Users\lj200\weatherAI\Temperature_Prediction_DeepLearning\models\LSTM\experiments\2026-08-16_08-14-02 (Standard Stacked LSTM regression model)\weather_lstm.keras
5878/5878 ━━━━━━━━━━━━━━━━━━━━ 59s 10ms/step - loss: 37.2815 - mae: 4.5719 - mse: 37.2815 - rmse: 6.1059 - val_loss: 82.7603 - val_mae: 7.4687 - val_mse: 82.7603 - val_rmse: 9.0973 - learning_rate: 0.0010
Epoch 2/100
5878/5878 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 26.1592 - mae: 3.9622 - mse: 26.1592 - rmse: 5.1126
Epoch 2: val_mae did not improve from 7.46868
5878/5878 ━━━━━━━━━━━━━━━━━━━━ 60s 10ms/step - loss: 25.9683 - mae: 3.8431 - mse: 25.9683 

Epoch 1/100
5872/5873 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 2.9199 - mae: 2.5648 - mse: 13.2106 - rmse: 3.6268
Epoch 1: val_mae improved from None to 2.05291, saving model to c:\Users\lj200\weatherAI\Temperature_Prediction_DeepLearning\models\LSTM\experiments\2026-08-16_10-02-06 (Residual Baseline)\weather_lstm.keras

Epoch 1: finished saving model to c:\Users\lj200\weatherAI\Temperature_Prediction_DeepLearning\models\LSTM\experiments\2026-08-16_10-02-06 (Residual Baseline)\weather_lstm.keras
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 150s 25ms/step - loss: 2.5085 - mae: 2.2772 - mse: 10.6398 - rmse: 3.2619 - val_loss: 2.1666 - val_mae: 2.0529 - val_mse: 8.1305 - val_rmse: 2.8514 - learning_rate: 3.0000e-04
Epoch 2/100
5872/5873 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 2.5020 - mae: 2.2762 - mse: 10.3129 - rmse: 3.2073
Epoch 2: val_mae did not improve from 2.05291
5873/5873 ━━━━━━━━━━━━━━━━━━━━ 187s 32ms/step - loss: 2.2530 - mae: 2.0997 - mse: 8.9372 - rmse: 2.9895 - val_loss: 2.1870 - val_m

,model,test_mse,test_mae_c,test_rmse_c,experiment_dir
0,Standard Stacked LSTM regression model,33.712316,4.639358,5.806231,c:\Users\lj200\weatherAI\Temperature_Predictio...
1,Residual Baseline,6.774556,1.776371,2.602798,c:\Users\lj200\weatherAI\Temperature_Predictio...
